# Targeted Advertising Mean-Field Control Benchmark

Reference: Meunier, Pham & Reisinger, discrete-space benchmarks, Section "Targeted advertising with social influence" (`files/reference/discrete_benchmarks.tex`), introduced by Motte & Pham. A company repeatedly advertises to increase its customer proportion while controlling expenditure.

**Model.** State space $\mathcal X=\{0,1\}$ (not-customer / customer), action space $\mathcal A=\{0,1\}$ (no ad / ad). Unlike every other benchmark in this repo, the mean-field interaction lives in the **transition kernel**, not the reward:
$$P(1\mid x,a,\mu) = \min\{\mu(1) + \kappa_\mathrm{ad}a,\, 1\}, \qquad r(x,a,\mu) = x - c_\mathrm{ad}a, \qquad g(x,\mu)=0,$$
independent of $x$: an advertisement raises *everyone's* conversion/retention probability by $\kappa_\mathrm{ad}$, and the current customer proportion $\mu(1)$ itself drives a positive social-influence effect. Rewards are discounted by $\gamma=0.5$ (truncating the source infinite-horizon problem to $T=5$).

**Policy.** Because only the aggregate advertising rate enters the population recursion, the optimal policy class is **individual-state-independent**: $\pi_\theta(1\mid t,x,\mu)=q_\theta(t/T,\mu(1))$ for every $x$. $q_\theta$ is a small 2-hidden-layer MLP (width 32, $\tanh$, sigmoid output, ~1.2k parameters) taking normalized time and the customer proportion.

**Unlike cybersecurity/distribution planning, this benchmark has a known closed-form structural benchmark**: the source model's stationary infinite-horizon optimal advertising probability $\hat q(p)$ (`Advertising.reference_policy`), a three-regime piecewise function of the customer proportion $p$. At this benchmark's parameters ($\kappa_\mathrm{ad}=0.2$, $c_\mathrm{ad}=0.15$, $\gamma=0.5$), it works out to
$$\hat q(p) = \begin{cases}1, & p<0.6 \\ (0.8-p)/0.2, & 0.6\le p<0.7 \\ 1, & 0.7\le p<0.85 \\ 0, & p\ge 0.85\end{cases}$$
a non-monotone rule this notebook compares the learned policy against directly.

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch

torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")

from configs.advertising import MID
from mfc.environments.advertising import AD, CUSTOMER, NO_AD, NOT_CUSTOMER, Advertising
from mfc.plotting import diagnostics as viz
from scripts.train import run_all
from scripts.test import (
    generalization_eval,
    gradient_diagnostics,
    load_runs,
    objective_gap,
    perturbation_coverage,
    reference_policy_fn,
    rollout,
    state_distribution,
)

## Configuration and budget

This notebook demonstrates the **mid** run tier from `configs/advertising.py`: one seed, the reference's horizon $T=5$, full training length (5000 iterations — the reference doesn't state a numeric training length for this benchmark; see `configs/advertising.py`'s module docstring for what this repo filled in and why), and the equal-parameters budget. Run `scripts/train.py --config main` separately for the full main-tier sweep (5 seeds, 10000 iterations).

In [ ]:
cfg = MID
print(f"algorithms:    {cfg.algorithms}")
print(f"lambdas:       {cfg.lambdas}  (simplex perturbation scale)")
print(f"epsilon:       {cfg.epsilon}  (logit perturbation scale, fixed per context.md)")
print(f"T={cfg.horizons[0]}")
print(f"seeds:         {cfg.seeds}")
print(f"B={cfg.B}, n_aux={cfg.n_aux}, sigma={cfg.sigma}, lr={cfg.lr}, n_train={cfg.n_train}")
print(f"mu0 ~ (1-p0,p0), p0~U([0.05,0.95]) during training; validation mu0={cfg.mu0_val}")

env_preview = Advertising()
print(f"kappa_ad={env_preview.config.kappa_ad}, c_ad={env_preview.config.c_ad}, gamma={env_preview.config.gamma}")

## Train (or load cached results)

Loads every saved run under `runs/advertising/mid/` for all three algorithms; trains first if none exist yet.

In [ ]:
env = Advertising()
runs_dir = ROOT / "runs" / "advertising" / "mid"

runs = []
for alg in cfg.algorithms:
    if not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_all("advertising", alg, "mid")
    runs += load_runs("advertising", alg, "mid")

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded; total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")
for r in sorted(runs, key=lambda r: (r["alg"], r["lam"] if r["lam"] is not None else -1)):
    tag = f"lambda={r['lam']}" if r["lam"] is not None else "(fixed epsilon)"
    print(f"  {r['alg']:<11} {tag:<16} seed={r['seed']}  elapsed={r['elapsed_seconds']:.1f}s  final validation J={r['validation_J'][-1].item():.4f}")

## Ground truth: the closed-form reference policy

The infinite-horizon optimal advertising rate $\hat q(p)$ (reference "Infinite-horizon reference policy"), and its induced trajectory from the validation initial law — the reference line every plot below compares against.

In [ ]:
from mfc.plotting.style import apply_style, color_for, new_figure, style_legend

ps = torch.linspace(0.0, 1.0, 201, dtype=env.dtype)
q_hat = env.reference_policy(ps)

fig, ax = new_figure()
ax.plot(ps.cpu(), q_hat.cpu(), color=color_for(0), linewidth=2, label="q_hat(p) (closed-form optimal)")
apply_style(ax, xlabel="customer proportion p", ylabel="advertising probability q_hat(p)", title="Closed-form infinite-horizon reference policy")
style_legend(ax)

mu0_val = torch.tensor(cfg.mu0_val, dtype=env.dtype, device=env.device)
reference_traj = state_distribution(env, reference_policy_fn(env), env.init_theta(), mu0_val, cfg.horizons[0])
print("customer proportion under q_hat, starting from mu0_val:", reference_traj[:, CUSTOMER].tolist())

## Evolution of the validation reward

The exact validation objective $J_T(\theta_m;\mu_0^\mathrm{val})$ every 10 training iterations, one line per simplex $\lambda$ plus reinforce and mfreinforce.

In [ ]:
fig, ax = viz.plot_validation_curve(runs)
ax.set_title("Validation objective during training", loc="left")

## Learned policy vs. the closed-form reference, across time

$q_\theta(t/T, p)$ for $\lambda=0.2$'s learned policy, at a few decision times $t$, against the stationary $\hat q(p)$. Since $\hat q$ is the *infinite-horizon* optimum, the learned finite-horizon policy is expected to diverge from it near the terminal boundary $t=T-1$ (no continuation value left to protect).

In [ ]:
by_lambda = {r["lam"]: r for r in runs if r["alg"] == "simplex"}  # mid has one seed per lambda
theta_02 = by_lambda[0.2]["theta_final"]

fig, ax = new_figure()
ax.plot(ps.cpu(), q_hat.cpu(), color="black", linestyle="--", linewidth=2, label="q_hat(p) (reference)")
for i, t in enumerate([0, cfg.horizons[0] // 2, cfg.horizons[0] - 1]):
    q_learned = torch.stack([env.policy_probs(theta_02, t, torch.tensor(NOT_CUSTOMER, device=env.device), torch.stack([1 - p, p]))[AD] for p in ps])
    ax.plot(ps.cpu(), q_learned.detach().cpu(), color=color_for(i + 1), linewidth=2, label=f"q_theta(t={t}, p)")
apply_style(ax, xlabel="customer proportion p", ylabel="advertising probability", title="Learned vs reference policy (lambda=0.2)")
style_legend(ax)

## Customer-proportion trajectory: learned vs reference

The exact population flow (customer proportion $\mu_t(1)$) from $\mu_0^\mathrm{val}$, under the learned policy ($\lambda=0.2$) against the closed-form reference policy.

In [ ]:
learned_flow = state_distribution(env, env.policy_probs, theta_02, mu0_val, cfg.horizons[0])
fig, ax = viz.plot_state_distribution(learned_flow, optimal_flow=reference_traj, state_labels=["not customer", "customer"])
ax.set_title("Customer-proportion flow (learned solid, reference dashed)", loc="left")

## $J^\lambda$ vs $J$ and gradient bias/variance

As elsewhere, empirical bias/variance of the simplex plug-in gradient estimator vs. the exact (autograd) gradient, summarized as $\|\text{bias}\|$/$\|\text{std}\|$ over theta (small here, ~1.2k parameters, but still not legible component-by-component).

In [ ]:
gaps, grad_diag = {}, {}
for lam, r in by_lambda.items():
    theta = r["theta_final"]
    gaps[lam] = objective_gap(env, env.policy_probs, theta, mu0_val, cfg.horizons[0], lam=lam, sigma=cfg.sigma, n_samples=2000)
    grad_diag[lam] = gradient_diagnostics(
        env, env.policy_probs, theta, mu0_val, cfg.horizons[0],
        lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=20,
    )

fig, ax = viz.plot_objective_gap(gaps)
ax.set_title("J vs J^lambda", loc="left")

bias_norm = {lam: grad_diag[lam]["bias"].norm().item() for lam in grad_diag}
std_norm = {lam: grad_diag[lam]["std"].norm().item() for lam in grad_diag}
fig, ax = viz.plot_horizon_scaling(bias_norm, xlabel="\u03bb", ylabel="norm over ~1.2k MLP parameters", label="||bias||", integer_xaxis=False)
viz.plot_horizon_scaling(std_norm, xlabel="\u03bb", label="||std||", color_index=1, integer_xaxis=False, ax=ax)
ax.set_title("Simplex gradient estimator: bias/std norms vs lambda", loc="left")

## Simplex-perturbation coverage: $d_{TV}(M^\lambda,\mu)\le\lambda$

Checked (as elsewhere) by direct sampling at a few representative population laws.

In [ ]:
coverage = perturbation_coverage(
    torch.stack([mu0_val, torch.tensor([0.9, 0.1], dtype=env.dtype), torch.tensor([0.5, 0.5], dtype=env.dtype)]),
    lam=0.2, sigma=cfg.sigma, n_samples=5000,
)
fig, ax = viz.plot_perturbation_coverage(coverage, lam=0.2, mu_labels=["mu0_val", "mostly non-customers", "uniform"])
ax.set_title("d_TV(M^lambda, mu) vs the lambda=0.2 bound", loc="left")

for r in coverage:
    assert r["within_bound"], "the perturbation theorem's bound should never be violated"
print("bound holds for all sampled mu (as guaranteed by the theorem).")

## Sample trajectories: learned vs reference policy

One sampled state trajectory under the learned policy ($\lambda=0.2$) and one under the closed-form reference policy, both from $\mu_0^\mathrm{val}$.

In [ ]:
learned_traj = rollout(env, env.policy_probs, theta_02, mu0_val, T=cfg.horizons[0], generator=torch.Generator(device=env.device).manual_seed(0))
reference_traj_sample = rollout(env, reference_policy_fn(env), env.init_theta(), mu0_val, T=cfg.horizons[0], generator=torch.Generator(device=env.device).manual_seed(1))
fig, ax = viz.plot_trajectories(learned_traj, reference_traj_sample)
ax.set_yticks([NOT_CUSTOMER, CUSTOMER])
ax.set_yticklabels(["not customer", "customer"])
ax.set_title("Sample trajectories (learned solid, reference dashed)", loc="left")

## Generalization without retraining

Evaluating the $\lambda=0.2$ learned $\theta$ exactly (no retraining) under different initial laws, a longer horizon, and model misspecification (shifted advertising efficiency/cost).

In [ ]:
from mfc.environments.advertising import AdvertisingConfig

scenarios = [
    {"name": "baseline (mu0_val)"},
    {"name": "mu0=mostly non-customers", "mu0": torch.tensor([0.9, 0.1], dtype=env.dtype, device=env.device)},
    {"name": "mu0=mostly customers", "mu0": torch.tensor([0.1, 0.9], dtype=env.dtype, device=env.device)},
    {"name": "T=10", "T": 10},
    {"name": "2x advertising efficiency", "env": Advertising(AdvertisingConfig(kappa_ad=0.4))},
    {"name": "2x advertising cost", "env": Advertising(AdvertisingConfig(c_ad=0.3))},
]
gen_results = generalization_eval(env, env.policy_probs, theta_02, mu0_val, cfg.horizons[0], scenarios)
fig, ax = viz.plot_generalization(gen_results)
ax.set_title("J under different scenarios (theta fixed, no retraining)", loc="left")

## Comparing the three algorithms

At `mid`'s single seed, final validation objective for each algorithm, and each one's distance from the closed-form reference at the validation initial law's horizon.

In [ ]:
by_alg = {}
for r in runs:
    by_alg.setdefault(r["alg"], []).append(r)

best_simplex = max(by_alg["simplex"], key=lambda r: r["validation_J"][-1].item())
print(f"best simplex lambda: {best_simplex['lam']}")
for alg in cfg.algorithms:
    r = best_simplex if alg == "simplex" else by_alg[alg][0]
    flow = state_distribution(env, env.policy_probs, r["theta_final"], mu0_val, cfg.horizons[0])
    flow_err = (flow[:, CUSTOMER] - reference_traj[:, CUSTOMER]).abs().mean().item()
    print(f"  {alg:<11} final validation J = {r['validation_J'][-1].item():.4f}   mean |p_t - p_t^ref| = {flow_err:.4f}")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")